In [1]:
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.model_selection import train_test_split
from sklearn import metrics
import os

In [2]:
df = pd.read_csv('data/data_ludwig.csv')

In [8]:
species = df.label.unique().tolist()

In [9]:
specie = {specie: index for index, specie in enumerate(species)}
df['label'] = df['label'].map(specie)

In [11]:
df['label'].nunique()

56

In [13]:
# Verificar que df es un DataFrame válido
print("Tipo de df:", type(df))
print("Columnas de df:", df.columns)

Tipo de df: <class 'pandas.core.frame.DataFrame'>
Columnas de df: Index(['label', 'Genus', 'Family', 'audio_path', 'Suborder',
       'Beak.Length_Culmen', 'Beak.Length_Nares', 'Beak.Width', 'Beak.Depth',
       'Tarsus.Length', 'Wing.Length', 'Kipps.Distance', 'Secondary1',
       'Hand-Wing.Index', 'Tail.Length', 'Mass', 'Habitat', 'Habitat.Density',
       'Migration', 'Trophic.Level', 'Trophic.Niche', 'Primary.Lifestyle',
       'Min.Latitude', 'Max.Latitude', 'Centroid.Latitude',
       'Centroid.Longitude', 'Range.Size', 'Species.Status'],
      dtype='object')


In [16]:
from pydub import AudioSegment

def convert_to_mono(input_path, output_path):
    """Convert stereo audio file to mono."""
    audio = AudioSegment.from_file(input_path)
    mono_audio = audio.set_channels(1)
    mono_audio.export(output_path, format="mp3")

def preprocess_audio_df(input_csv, output_csv, output_dir):
    """Preprocess audio files, converting them to mono and storing path in CSV."""
    os.makedirs(output_dir, exist_ok=True)
    df = pd.read_csv(input_csv)
    converted_paths = []

    for index, row in df.iterrows():
        original_path = row['audio_path']
        output_path = os.path.join(output_dir, os.path.basename(original_path))
        
        # Check if the file already exists
        if not os.path.isfile(output_path):
            try:
                convert_to_mono(original_path, output_path)
                converted_paths.append(output_path)
            except Exception as e:
                print(f"Error converting {original_path}: {e}")
                converted_paths.append(None)
        else:
            print(f"File already exists: {output_path}")
            converted_paths.append(output_path)

    df['audio_path'] = converted_paths
    df.to_csv(output_csv, index=False)

# Define paths
input_csv = './data/data_ludwig.csv'
output_csv = './data/data_prueba_preprocessed.csv'
output_dir = './transfor_songs/converted_audio_files'

In [ ]:
preprocess_audio_df(input_csv, output_csv, output_dir)

In [ ]:
df = pd.read_csv('./data/data_prueba_preprocessed.csv')
df['label'] = df['label'].map(specie)

df.to_csv('./data/data_prueba_preprocessed.csv', index=False)